# Project 6: Privacy-Preserving Machine Learning
Bach Nguyen, Son Nguyen

This project explore application of the Differentially Private ID3 algorithm on the Adult Census Income dataset. 

In [ ]:
import os
import os.path
import pandas as pd
import numpy as np

## Loading in the data

In [ ]:
np.random.seed(42)

datadir = "data"
data = os.path.join(datadir, "adult.data")
df = pd.read_csv(data)
df.columns = ["age", "workclass", "fnlwgt", "education", "education-num", 
              "marital-status", "occupation", "relationship", "race", "sex", 
              "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"]

# Dropping education-num
df.drop(columns=["education-num"], inplace=True)

## Helper Function

In [ ]:
def find_entropy_split(D, a, epsilon, label_col):
    """
    Differentially private entropy of the split on attribute `a`.

    Input:
        D (pd.DataFrame): Dataset containing feature columns and a label column.
        a (str): Name of the feature/attribute to split on.
        epsilon (float): Privacy parameter ε used for Laplace noise.
        label_col (str): Name of the label column in D.

    Output:
        float: Noisy expected entropy after splitting on attribute `a`.
    """
    n = len(D)

    tot = 0.0

    # For each unique value j of attribute a
    for j in D[a].unique():
        D_j = D[D[a] == j]
        count_D_j = len(D_j) + np.random.laplace(0.0, 1.0/epsilon)

        subtree_entropy = 0.0

        for i in D[label_col].unique():
            true_count = int(sum(D_j[label_col] == i))
            count = true_count + np.random.laplace(loc=0.0, scale=1.0/epsilon)

            pi = count / count_D_j
            subtree_entropy -= pi * np.log2(pi)

        tot += subtree_entropy * (count_D_j / n)

    return tot